# Tractography masterclass: from a whole-brain tractogram to bundle-specific microstructure

[Click here to Open in Colab](https://colab.research.google.com/github/chiaramff/Tractography_IST_Masterclass/blob/main/tractography_masterclass.ipynb)

In this notebook:

1 - from whole brain tractogram to bundles

2 - comparison of different methods to extract bundles

3 - extract whole-brain FA measures

4 - extract along-tract FA measures

5 - generate whole-brain connectivity matrix


| Part | What we do |
|---|---|
| 1 | Get the data (public FreeSurfer TRACULA tutorial subject) + whole-brain tractograms |
| 2 | Extract the same 8 bundles with three different approaches: **ROI-based filtering with `tckedit`** (MRtrix3), **TRACULA** (FreeSurfer) and **RecoBundles** (DIPY) |
| 3 | Look at the bundles interactively (pick a tract, rotate, zoom, scroll through slices) |
| 4 | Extract **mean FA** per bundle with TRACULA, MRtrix3 and DIPY: unweighted *vs* weighted |
| 5 | Extract **along-tract profiles** step by step |
| 6 | Generate whole-brain connectivity matrix

DISCLOSURE! This is not meant to be the optimal way to process your data, rather to show different approaches to do the same thing and warn you about how that affects the results!

**Bundles:** left/right arcuate fasciculus (AF), left/right corticospinal tract (CST), left/right inferior longitudinal fasciculus (ILF), left/right uncinate fasciculus (UF).

> **Running time.** Everything except RecoBundles should take a few seconds per bundle. RecoBundles takes ~1–2 min per bundle the first time; results are cached in `outputs/` so re-running is fast.

## 0. Setup

The notebook needs `dipy`, `nibabel`, `plotly`, `scikit-image`, `pandas`, `matplotlib`, and **MRtrix3** (command-line tools on the `PATH`).
`environment.yml` in this repository installs all of them.
**FreeSurfer** is optional (TRACULA outputs are precomputed and if `mri_extract_label` is not on the $PATH nibabel is used instead to make the ROIs).
Plotting and other helper code is in `helpers.py`, next to this notebook, so the cells below show only the analysis steps.

**Google Colab:** the next cell sets everything up when the notebook runs on Colab (copies this repository, installs the Python packages and MRtrix3; ~3 min). Anywhere else it does nothing.

**Running it on your own computer:** install everything with `environment.yml` (see the README) and start Jupyter from this folder. If MRtrix3, FreeSurfer or the tutorial data are somewhere the notebook can't find them, add their paths in the "Optional: your paths" cell.

In [ ]:
# --- Google Colab: set up the environment (does nothing anywhere else) ------------------------------
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("helpers.py"):                              # copy this repository
        subprocess.run(["git", "clone", "-q", "https://github.com/chiaramff/Tractography_IST_Masterclass.git", "/content/masterclass"], check=True)
        os.chdir("/content/masterclass")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "dipy==1.9.*", "plotly", "scikit-image"], check=True)
    if not os.path.exists("/content/mrtrix/bin/tckgen"):              # MRtrix3 with micromamba
        subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba",
                       shell=True, check=True, cwd="/content")
        subprocess.run(["/content/bin/micromamba", "create", "-y", "-q", "-p", "/content/mrtrix",
                        "-c", "conda-forge", "-c", "mrtrix3", "mrtrix3"], check=True)
    os.environ["MRTRIX_DIR"] = "/content/mrtrix/bin"

In [ ]:
# --- Optional: your paths ------------------------------------------------------------------------------
# Only needed if something is not found automatically (e.g. MRtrix3 installed outside the conda environment).
# Leave "" to skip a line.
import os
from pathlib import Path

MY_PATHS = {
    "MRTRIX_DIR": "",             # folder with the MRtrix3 commands (tckgen, tckedit, ...), e.g. "/opt/mrtrix3/bin"
    "FREESURFER_HOME": "",        # FreeSurfer installation, e.g. "/usr/local/freesurfer/7.4.1" (optional)
    "MRTRIX_LABELCONVERT_DIR": "",  # only if MRtrix3 runs from a container: a folder with fs_default.txt (section 7)
    "TRACULA_TUTORIAL_DIR": "",   # an existing copy of the TRACULA tutorial data, to skip the download
}

if Path("local_setup.py").exists():          # personal settings kept outside the notebook (not in the repository)
    exec(Path("local_setup.py").read_text())
for name, value in MY_PATHS.items():
    if value:
        os.environ[name] = value
if os.environ.get("FREESURFER_HOME"):        # put FreeSurfer's commands on the PATH
    os.environ["PATH"] = os.environ["FREESURFER_HOME"] + "/bin" + os.pathsep + os.environ["PATH"]

In [ ]:
import os
from pathlib import Path

# ---- configuration ----------------------------------------------------------------------------
ROOT = Path.cwd()                      # everything is read from / written to this folder
DATA = ROOT / "data"                   # downloaded inputs
OUT = ROOT / "outputs"                 # everything this notebook produces
SUB = "sub-BANDA001"                   # name of the TRACULA Tutorial subject

# TRACULA tutorial data (FreeSurfer): the files we use (~4 MB) are downloaded from this repository's release.
# If you already have the tutorial data locally, set TRACULA_TUTORIAL_DIR (cell above) to skip the download:
TRACULA_LOCAL = os.environ.get("TRACULA_TUTORIAL_DIR")          # e.g. ".../tracula_tutorial"

# Precomputed Whole-brain tractograms (~170 MB each): probabilistic (iFOD2) and deterministic (SD_STREAM)
WHOLEBRAIN_FILE = ROOT / "derivatives" / SUB / "tractography" / "wholebrain.tck"
WHOLEBRAIN_DET_FILE = ROOT / "derivatives" / SUB / "tractography" / "wholebrain_det.tck"
RELEASE_URL = os.environ.get("RELEASE_URL", "https://github.com/chiaramff/Tractography_IST_Masterclass/releases/download/v1.0")   # tutorial data + tractograms

# keep all caches / temp files inside the project folder
os.environ.setdefault("DIPY_HOME", str(DATA / "dipy_home"))          # RecoBundles atlas download
os.environ.setdefault("MRTRIX_TMPFILE_DIR", str(OUT / "tmp"))
if os.environ.get("MRTRIX_DIR"):                                     # MRtrix3 not on PATH? set MRTRIX_DIR
    os.environ["PATH"] = os.environ["MRTRIX_DIR"] + os.pathsep + os.environ["PATH"]
for d in (DATA, OUT, OUT / "tmp"):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import shutil, time, warnings
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import nibabel as nib

from dipy.io.streamline import load_tractogram, load_trk, save_tractogram
from dipy.io.stateful_tractogram import StatefulTractogram, Space
from dipy.tracking.streamline import Streamlines, set_number_of_points, values_from_volume, length, transform_streamlines
from dipy.tracking.utils import density_map
from dipy.stats.analysis import afq_profile, gaussian_weights, assignment_map
import dipy

from helpers import (run, fs_labels, extract_label, endpoint_roi, fetch_tutorial_data, save_tck, save_mm, load_mm,
                     atlas_name, read_pathstats, read_byvoxel, above_robust_max, bundle_length, fa_table,
                     plot_wholebrain, plot_rois, plot_side_by_side, plot_tracula, plot_atlas_bundles, plot_slr, plot_reco_thresholds,
                     plot_bundles_3d, plot_side_by_side, slice_scroller, plot_mean_fa, plot_mrtrix_vs_dipy,
                     orient, profile_tracula, plot_orientation, plot_profile_steps, plot_profiles,
                     find_lut, read_node_names, plot_connectome, plot_edges)

warnings.filterwarnings("ignore", category=UserWarning)
print("dipy", dipy.__version__)
print("MRtrix3:", shutil.which("tckedit") or "NOT FOUND - set MRTRIX_DIR or install mrtrix3")
print("FreeSurfer:", shutil.which("mri_extract_label") or "not found (optional: ROIs are made in Python instead)")

In [ ]:
# Bundles
BUNDLES = ["lh.af", "rh.af", "lh.cst", "rh.cst", "lh.ilf", "rh.ilf", "lh.uf", "rh.uf"]
METHODS = ["tckedit", "TRACULA", "RecoBundles"]
COLORS = {"tckedit": "#1baf7a", "TRACULA": "#2a78d6", "RecoBundles": "#eb6834"}   # same colour = same method, everywhere

## 1. Data

We use **`sub-BANDA001`** from the [FreeSurfer TRACULA tutorial](https://surfer.nmr.mgh.harvard.edu/fswiki/FsTutorial/Tracula)
([BANDA study](https://pmc.ncbi.nlm.nih.gov/articles/PMC7199015/), multi-shell, b = 1500/3000 s/mm², 1.5 mm isotropic). From the full TRACULA output we use:

* `dmri/dtifit_FA.nii.gz`: the FA map.
* `dlabel/diff/aparc+aseg.bbr.nii.gz`: FreeSurfer parcellation already registered to diffusion space.
* `dpath/<bundle>_avg16_syn_bbr/`: TRACULA bundle reconstructions.

Everything is in the **same diffusion space** (140 × 140 × 92 voxels, 1.5 mm).

Files we use are downloaded from this repository's release, together with the two whole-brain tractograms. Downloads happen only the first time.

In [ ]:
TUTORIAL = fetch_tutorial_data(DATA, RELEASE_URL, TRACULA_LOCAL)        # downloads once, then reuses data/
TRC = TUTORIAL / "trc" / SUB
FA_FILE = TRC / "dmri" / "dtifit_FA.nii.gz"
APARC_FILE = TRC / "dlabel" / "diff" / "aparc+aseg.bbr.nii.gz"
BRAINMASK_FILE = TRC / "dlabel" / "diff" / "lowb_brain_mask.nii.gz"
tracula_dir = {b: TRC / "dpath" / f"{b}_avg16_syn_bbr" for b in BUNDLES}

for f in (WHOLEBRAIN_FILE, WHOLEBRAIN_DET_FILE):
    if not f.exists():
        f.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", f.name, "...")
        urlretrieve(f"{RELEASE_URL}/{f.name}", f)

# load FA and the brain mask (for the 3D views)
fa_img = nib.load(FA_FILE)
FA, AFFINE = fa_img.get_fdata(), fa_img.affine
VOXEL_MM = fa_img.header.get_zooms()[0]                        # 1.5 mm
VOXEL_MM3 = np.prod(fa_img.header.get_zooms()[:3])             # 1.5 x 1.5 x 1.5 mm = 3.375 mm³
brain = nib.load(BRAINMASK_FILE).get_fdata()
print("FA:", FA.shape, fa_img.header.get_zooms(), "orientation", "".join(nib.aff2axcodes(AFFINE)))

### The whole-brain tractogram

The whole-brain tractograms were generated with MRtrix3. (`scripts/01_whole_brain_tractogram.sh`):
```bash
dwi2response dhollander dwi.mif wm.txt gm.txt csf.txt -mask mask.mif
dwi2fod msmt_csd dwi.mif -mask mask.mif wm.txt wmfod.mif gm.txt gm.mif csf.txt csf.mif
mtnormalise wmfod.mif wmfod_norm.mif gm.mif gm_norm.mif csf.mif csf_norm.mif -mask mask.mif
5ttgen freesurfer aparc+aseg.bbr.nii.gz 5tt.mif -nocrop          # ACT: tissue types from FreeSurfer
5tt2gmwmi 5tt.mif gmwmi.mif                                      # seeds at the grey/white matter interface
tckgen wmfod_norm.mif tracks_2M.tck -algorithm iFOD2 -backtrack -step 0.75 -angle 45 -act 5tt.mif -crop_at_gmwmi \
       -seed_gmwmi gmwmi.mif -select 2000000 -minlength 20 -maxlength 250 -cutoff 0.06
tcksift tracks_2M.tck wmfod_norm.mif wholebrain.tck -act 5tt.mif -term_number 200000

# deterministic: same options, no -backtrack (it only works for probabilistic algorithms)
tckgen wmfod_norm.mif tracks_2M_det.tck -algorithm SD_STREAM -step 0.75 -angle 45 -act 5tt.mif -crop_at_gmwmi \
       -seed_gmwmi gmwmi.mif -select 2000000 -minlength 20 -maxlength 250 -cutoff 0.06
tcksift tracks_2M_det.tck wmfod_norm.mif wholebrain_det.tck -act 5tt.mif -term_number 200000
```
* **Probabilistic** (iFOD2, `wholebrain.tck`): at every step, draw a direction from the FOD around the current direction. Streamlines can follow fanning and crossing fibres, and the bundle spreads more.
* **Deterministic** (SD_STREAM, `wholebrain_det.tck`): at every step, follow the FOD peak closest to the current direction. The same seed always gives the same streamline.

In [ ]:
# Explore the tractogram by streamline length (slider).
wb = load_tractogram(str(WHOLEBRAIN_FILE), str(FA_FILE), bbox_valid_check=False).streamlines
print(f"{len(wb):,} streamlines, median length {np.median(length(wb)):.0f} mm")

plot_wholebrain(wb)
# Colour = direction (red: left-right, green: anterior-posterior, blue: superior-inferior).

## 2. From whole brain to bundles: three approaches

| | **ROI-based (`tckedit`)** | **TRACULA** | **RecoBundles** |
|---|---|---|---|
| Input | whole-brain tractogram + ROIs | DWI + FreeSurfer recon-all output | whole-brain tractogram + bundle atlas |
| Prior knowledge | the ROIs *you* define (start/end, exclusions) | anatomical neighbourhood priors learned from manually annotated training subjects | shape of model bundles (HCP842 atlas, MNI) |
| Output | subset of the whole-brain streamlines (trk) | probability distribution of the path (`path.pd.nii.gz`)  | subset of the whole-brain streamlines (trk) |
| Main failure mode | depends on ROI choice; endpoint precision | depends on segmentation + atlas bundle definition | depends on registration + atlas bundle definition |

Note: **`tckedit` and RecoBundles both select streamlines from the same whole-brain tractogram** and are therefore limited by the quality of the data, whereas **TRACULA** leverages global tractography to input prior anatomical information in the reconstruction of the pathway.

In [ ]:
bundles = {m: {} for m in METHODS}
BUNDLE_DIR = {m: OUT / "bundles" / m for m in METHODS}
for d in BUNDLE_DIR.values():
    d.mkdir(parents=True, exist_ok=True)

### 2.1 ROI-based filtering with `tckedit` (MRtrix3)

The classic ROI-based approach: keep streamlines that **connect region A to region B** and **don't enter exclusion regions**.
We make the ROIs from the FreeSurfer `aparc+aseg` that is already in diffusion space, with FreeSurfer's `mri_extract_label` (one command per ROI).

| bundle | ROI A  | ROI B  | exclude ROIs (besides the other hemisphere) |
|---|---|---|---|
| AF | pars opercularis/triangularis, precentral, caudal middle frontal | superior/middle/inferior temporal, banks STS | insula, occipital cortex, deep grey matter, cerebellum, brainstem |
| CST | precentral, paracentral gyri | brainstem | cerebellum |
| ILF | lateral occipital, lingual, cuneus, pericalcarine | temporal pole, inferior/middle temporal | frontal cortex, thalamus, brainstem, cerebellum |
| UF | lateral/medial orbitofrontal, pars orbitalis, frontal pole | temporal pole, entorhinal | occipital & parietal cortex, thalamus, brainstem, cerebellum |

Practical details that matter:
* **Grow the endpoint ROIs by one voxel, into white matter only.** With ACT + `-crop_at_gmwmi`, streamlines stop at the grey/white matter interface, often in the white-matter voxel just below the cortex, and would miss the cortical label. 
* ** If A and B are close or overlap.** Neighbouring labels touch, so after growing they can overlap, and a streamline with both ends in the overlap would count as connecting A and B. We remove the shared voxels from both.
* We can set **minimum lengths** (e.g, `tckedit -minlength 60 for ILF`) to limit shorter "spourious" streamlines. 
* Use **two passes**: (1) `-include A -include B -ends_only` (the *endpoints* must be in A and B), then (2) `-exclude X` on the *whole* streamline. With a single command, `-ends_only` would also apply to the exclusion ROI.
* Remove A and B from the exclusion mask (`mrcalc`) so the masks never overlap.

NOTE: ** We are selectng end cortical ROIs here, but often a better way to define bundles is to place way point in WM regions

In [ ]:
# Extract ROIs from FreeSurfer segmentation; fs_labels() turns names into label numbers from FreeSurferColorLUT.txt
OCCIPITAL = ["lateraloccipital", "lingual", "cuneus", "pericalcarine"]
FRONTAL = ["superiorfrontal", "rostralmiddlefrontal", "caudalmiddlefrontal", "lateralorbitofrontal",
           "medialorbitofrontal", "parsorbitalis", "parstriangularis", "parsopercularis", "frontalpole", "precentral"]
DEEP_GM = ["thalamus", "caudate", "putamen", "pallidum"]
CEREBELLUM = ["cerebellum_wm", "cerebellum_ctx"]
ROI_DEF = {
    "af":  dict(A=["parsopercularis", "parstriangularis", "precentral", "caudalmiddlefrontal"],
                B=["superiortemporal", "middletemporal", "inferiortemporal", "bankssts"],
                X=["insula", *OCCIPITAL, *DEEP_GM, *CEREBELLUM, "brainstem"]),
    "cst": dict(A=["precentral", "paracentral"],
                B=["brainstem"],
                X=CEREBELLUM),
    "ilf": dict(A=OCCIPITAL,
                B=["temporalpole", "inferiortemporal", "middletemporal"],
                X=[*FRONTAL, "thalamus", *CEREBELLUM, "brainstem"]),
    "uf":  dict(A=["lateralorbitofrontal", "medialorbitofrontal", "parsorbitalis", "frontalpole"],
                B=["temporalpole", "entorhinal"],
                X=[*OCCIPITAL, "superiorparietal", "inferiorparietal", "supramarginal", "thalamus", *CEREBELLUM, "brainstem"]),
}

roi_dir = OUT / "rois"; roi_dir.mkdir(exist_ok=True)
WM = {h: roi_dir / f"{h}.wm.nii.gz" for h in ("lh", "rh")}   # endpoint ROIs may only grow into white matter
for h in WM:
    extract_label(APARC_FILE, fs_labels(h, ["cerebral_wm"]), WM[h], show=(h == "lh"))

for b in BUNDLES:
    h, t = b.split(".")
    other = "rh" if h == "lh" else "lh"
    show = b == BUNDLES[0]                                   # echo the commands for the first bundle
    A_all, B_all, A, B, X_all, X = (roi_dir / f"{b}_{k}.nii.gz" for k in ("A_all", "B_all", "A", "B", "X_all", "X"))
    endpoint_roi(APARC_FILE, fs_labels(h, ROI_DEF[t]["A"]), WM[h], A_all, show)
    endpoint_roi(APARC_FILE, fs_labels(h, ROI_DEF[t]["B"]), WM[h], B_all, show)
    run(["mrcalc", A_all, B_all, "-not", "-and", A, "-force", "-quiet"], show)      # A and B share no voxels
    run(["mrcalc", B_all, A_all, "-not", "-and", B, "-force", "-quiet"], show)
    extract_label(APARC_FILE, fs_labels(h, ROI_DEF[t]["X"]) + fs_labels(other, ["hemisphere"]), X_all, show=show)
    run(["mrcalc", X_all, A, B, "-or", "-not", "-and", X, "-force", "-quiet"], show)   # X without A and B
print("ROIs written to", roi_dir.relative_to(ROOT))
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print("NOTE: there are fancier ways to grow ROIs in WM, like using normals to expand along the white matter surface")

In [ ]:
#use tckedit to extract bundles
MIN_LENGTH = {"ilf": 60, "uf": 50}                           # mm, to exclude short fibres
for b in BUNDLES:
    ends = OUT / "tmp" / f"{b}_ends.tck"
    out = BUNDLE_DIR["tckedit"] / f"{b}.tck"
    #1 inclusion ROIs: keep only streamlines that connect both A and B regions
    run(["tckedit", WHOLEBRAIN_FILE, ends, "-include", roi_dir / f"{b}_A.nii.gz",
         "-include", roi_dir / f"{b}_B.nii.gz", "-ends_only", "-force", "-quiet"], show=(b == BUNDLES[0]))
    t = b.split(".")[1]
    min_len = ["-minlength", MIN_LENGTH[t]] if t in MIN_LENGTH else []
    #2 exclusion ROI: remove streamlines passing through X region
    run(["tckedit", ends, out, "-exclude", roi_dir / f"{b}_X.nii.gz", *min_len, "-force", "-quiet"],
        show=(b == BUNDLES[0] or b == "lh.ilf"))
    bundles["tckedit"][b] = load_tractogram(str(out), str(FA_FILE), bbox_valid_check=False).streamlines
    n_ends = int(run(["tckinfo", ends, "-count"], show=False).split()[-1])
    print(f"{b:7s}: {n_ends:5d} connect A-B  ->  {len(bundles['tckedit'][b]):5d} after exclusion"
          + (f" and -minlength {MIN_LENGTH[t]}" if t in MIN_LENGTH else ""))

print("Note the left UF: only about 30 streamlines pass, so its mean FA and profile below rest on a small sample.")

Let's look at the ROIs of the **left ILF**: A (occipital, green), B (temporal, blue), the exclusion ROI X in the same hemisphere (faint red), and the streamlines that go through A and B. A and B **meet along the occipito-temporal border**. Short U-fibres crossing that border also "connect" A and B, which is why here we set`-minlength 60` for ILF (optional!).

In [ ]:
plot_rois("lh.ilf", roi_dir, APARC_FILE, bundles["tckedit"]["lh.ilf"], brain, AFFINE)

#### Deterministic vs probabilistic tractography (same ROIs)

The bundles above come from the **probabilistic** whole-brain tractogram (iFOD2). We now run exactly the same `tckedit` step on the **deterministic**.

In [ ]:
# the same ROIs and tckedit passes, on the deterministic tractogram
DET_DIR = OUT / "bundles" / "tckedit_det"; DET_DIR.mkdir(parents=True, exist_ok=True)
bundles_det = {}
for b in BUNDLES:
    ends, out = OUT / "tmp" / f"{b}_ends_det.tck", DET_DIR / f"{b}.tck"
    run(["tckedit", WHOLEBRAIN_DET_FILE, ends, "-include", roi_dir / f"{b}_A.nii.gz",
         "-include", roi_dir / f"{b}_B.nii.gz", "-ends_only", "-force", "-quiet"], show=(b == BUNDLES[0]))
    t = b.split(".")[1]
    min_len = ["-minlength", MIN_LENGTH[t]] if t in MIN_LENGTH else []
    run(["tckedit", ends, out, "-exclude", roi_dir / f"{b}_X.nii.gz", *min_len, "-force", "-quiet"], show=(b == BUNDLES[0]))
    bundles_det[b] = load_tractogram(str(out), str(FA_FILE), bbox_valid_check=False).streamlines
    print(f"{b:7s}: {len(bundles_det[b]):5d} streamlines (probabilistic: {len(bundles['tckedit'][b])})")

In [ ]:
# Probabilistic vs Deterministic
#  streamlines count, length, and volume (every voxel the bundle touches, from a tckmap density map)
TRACKING = {"iFOD2": bundles["tckedit"], "SD_STREAM": bundles_det}
rows = []
for b in BUNDLES:
    for name, tck_dir in (("iFOD2", BUNDLE_DIR["tckedit"]), ("SD_STREAM", DET_DIR)):
        sl = TRACKING[name][b]
        density_file = OUT / "tmp" / f"{name}_{b}_density.nii.gz"
        run(["tckmap", tck_dir / f"{b}.tck", density_file, "-template", FA_FILE, "-upsample", 2, "-force", "-quiet"],
            show=(name, b) == ("SD_STREAM", "lh.af"))
        rows.append(dict(bundle=b, tracking=name, n_streamlines=len(sl),
                         median_length_mm=np.median(length(sl)) if len(sl) else np.nan,
                         volume_mm3=(nib.load(density_file).get_fdata() > 0).sum() * VOXEL_MM3))
det_df = pd.DataFrame(rows)
det_df.pivot(index="bundle", columns="tracking").reindex(columns=["iFOD2", "SD_STREAM"], level=1).loc[BUNDLES].round(
    {"n_streamlines": 0, "median_length_mm": 0, "volume_mm3": -1})

In [ ]:
#Visuzalize Deterministic and Probabilistic Bundles
plot_side_by_side(TRACKING, None, brain, AFFINE, {"iFOD2": COLORS["tckedit"], "SD_STREAM": "#7b4fd6"})

**Observations**
* Deterministic bundles look "cleaner", less fanning or fuzziness
* **More streamlines does not mean a bigger bundle.** For some bundles SD_STREAM gives *more* streamlines than iFOD2 but a smaller volume (more compact profile).
* The probabilistic bundles reach more of the cortex, but also contain more streamlines that may be false positives.
(From here on, the notebook uses the probabilistic bundles.)

### 2.2 TRACULA

TRACULA (Yendiki et al. 2011; Maffei et al. 2021) is run with three commands and a configuration file:

```bash
trac-all -prep -c dmrirc.masterclass    # preprocessing, registration, priors (FLS TOPUP+EDDY, ANTS, DTIFIT)
trac-all -bedp -c dmrirc.masterclass    # ball-and-stick model (FSL bedpostX)
trac-all -path -c dmrirc.masterclass    # pathway reconstruction (+ stats)
```

The configuration file for this subject is in this repository, `tracula/dmrirc.masterclass`. It points to the files of the TRACULA tutorial data: to run TRACULA yourself, set `TUTORIAL_DATA` at the top of the file to where you unpacked the tutorial data.
NOTE: You can also create your config file directly from a BIDS folder:
dmri_bids_config  --in <bids_directory>  --c <configfile>

For more details: https://surfer.nmr.mgh.harvard.edu/fswiki/dmrirc.

In [ ]:
#Let's look at the config file 
print((ROOT / "tracula" / "dmrirc.masterclass").read_text())

Running all 3 steps can take hours, so we use the outputs from the tutorial. For each bundle, the output folder `dpath/<bundle>_avg16_syn_bbr/` contains:

* `path.pd.nii.gz`: **posterior distribution** of the path, a volume.
* `path.pd.trk`: the **path samples** themselves (7500 MCMC samples, every 5th kept: 1,500 per bundle). Each sample is a chain of neighbouring voxels, not a tractography streamline; we only use them for display (the `path.pd.trk` entry in the 3D views).
* `pathstats.overall.txt`: whole-tract stats, computed by `dmri_pathstats`:
  * `Volume`, `FA_Avg`: on all path voxels, `FA_Avg_Weight`: on the voxels where `path.pd` is above **20% of its robust maximum** 
* `pathstats.byvoxel.txt`: along-tract stats, one row per voxel of the reference path (same number of rows in every subject)

TRACULA also fits the **tensor model** during `trac-all -prep`: FSL `dtifit` on the corrected DWIs (`dmri/data.nii.gz`), written to `dmri/dtifit_*` (FA, MD, L1–L3, V1–V3, S0, MO). We use the FA map in the notebook, and the AD/RD/MD/FA values in `pathstats` come from these maps. 

The cell below loads TRACULA's outputs and shows `pathstats.overall.txt` for the left AF:

In [ ]:
#Load TRACULA OUTPUTS
pd_vol = {b: nib.load(tracula_dir[b] / "path.pd.nii.gz").get_fdata() for b in BUNDLES}
for b in BUNDLES:
    bundles["TRACULA"][b] = Streamlines(nib.streamlines.load(str(tracula_dir[b] / "path.pd.trk")).streamlines)

print((tracula_dir["lh.af"] / "pathstats.overall.txt").read_text())

In [ ]:
# Along-tract - pathstats.byvoxel.txt
# one row per voxel of the reference path: voxel coordinates, the measures in that voxel,
# and (_Avg) their mean over the path samples at their closest point
read_byvoxel(tracula_dir["lh.af"]).head(10)

TRACULA reonstruction for the 8 bundles: path distribution as a surface (at 10% of its maximum). Click a bundle in the legend to hide or show it. The legend shows each bundle's `FA_Avg_Weight` from `pathstats.overall.txt` (section 4 explains the different FA averages).

In [ ]:
plot_tracula(pd_vol, brain, AFFINE, tracula_dir)      # hover: FA_Avg_Weight from pathstats

### 2.3 RecoBundles (DIPY)

RecoBundles (Garyfallidis et al. 2018) recognizes bundles by **shape similarity** to model bundles from an atlas.
We use the HCP842 atlas (Yeh et al. 2018; 80 bundles in MNI space), downloaded by DIPY (~200 MB, stored in `data/dipy_home/`).

Steps:
1. **Streamline-based linear registration (SLR)** of the subject's whole-brain tractogram to the atlas whole-brain tractogram (affine, no images involved).

2. For each model bundle, RecoBundles **pre-selects** subject clusters near the model (`reduction_thr`), **locally re-registers** them to the model (`slr=True`) and **prunes** streamlines further than `pruning_thr` (mm, minimum-direct-flip distance) from the model.

3. Output is "recognized" bundles in  atlas space (here MNI) and the indices of those streamlines in original native tractogram ('labels')

We follow the DIPY tutorial [Automatic Fiber Bundle Extraction with RecoBundles](https://docs.dipy.org/1.9.0/examples_built/segmentation/bundle_extraction.html) with its default settings (`reduction_thr=15`, `pruning_thr=7`). The tutorial also shows an optional second pass, `rb.refine(...)`, which re-registers and prunes the recognized bundle again (e.g. `pruning_thr=6`) to remove more stray streamlines; we don't run it here.

In [ ]:
from dipy.data.fetcher import fetch_bundle_atlas_hcp842, get_bundle_atlas_hcp842
from dipy.align.streamlinear import whole_brain_slr
from dipy.segment.bundles import RecoBundles

fetch_bundle_atlas_hcp842()                                           # downloads the atlas once (~200 MB)
atlas_file, model_glob = get_bundle_atlas_hcp842()
models = {b: load_trk(model_glob.replace("*", atlas_name(b)), "same", bbox_valid_check=False).streamlines
          for b in BUNDLES}                                           # model bundles, in MNI space

RECO = BUNDLE_DIR["RecoBundles"]
reco_files = {b: RECO / f"{b}.tck" for b in BUNDLES}
slr_files = dict(xfm=RECO / "slr_xfm.txt", atlas_centroids=RECO / "slr_atlas_centroids.tck",
                 subject_centroids=RECO / "slr_subject_centroids.tck")
if all(f.exists() for f in [*reco_files.values(), *slr_files.values()]):
    print("loading cached RecoBundles results from", RECO.relative_to(ROOT))
    for b, f in reco_files.items():
        bundles["RecoBundles"][b] = load_tractogram(str(f), str(FA_FILE), bbox_valid_check=False).streamlines
    xfm = np.loadtxt(slr_files["xfm"])
    atlas_centroids, subject_centroids = load_mm(slr_files["atlas_centroids"]), load_mm(slr_files["subject_centroids"])
else:
    atlas = load_trk(atlas_file, "same", bbox_valid_check=False).streamlines

    # 1. whole-brain SLR: subject -> MNI atlas (~1 min). It registers QuickBundles cluster centroids of the two
    #    tractograms (resampled to 20 points), not all streamlines; subject_centroids are BEFORE registration.
    moved, xfm, atlas_centroids, subject_centroids = whole_brain_slr(atlas, wb, x0="affine", verbose=False, progressive=True,
                                                                     rng=np.random.RandomState(1984))
    np.savetxt(slr_files["xfm"], xfm)
    save_mm(atlas_centroids, slr_files["atlas_centroids"]); save_mm(subject_centroids, slr_files["subject_centroids"])

    # 2. recognise each bundle (~1-2 min each)
    rb = RecoBundles(moved, verbose=False, rng=np.random.RandomState(2001))
    for b in BUNDLES:
        recognized_mni, labels = rb.recognize(model_bundle=models[b], model_clust_thr=0.1, reduction_thr=15, pruning_thr=7,
                                              reduction_distance="mdf", pruning_distance="mdf", slr=True)
        # recognized_mni: the bundle in atlas (MNI) space; labels: indices into our tractogram
        bundles["RecoBundles"][b] = wb[labels]                        # 3. the same streamlines in native space
        save_tck(bundles["RecoBundles"][b], reco_files[b], fa_img)
        print(f"{b:7s} ({atlas_name(b)}): {len(labels)} streamlines")

**The atlas.** The 8 HCP842 model bundles we look for, in MNI space:

In [ ]:
plot_atlas_bundles(models)

**Step 1: whole-brain registration (SLR).** SLR does not register all 200,000 streamlines: it first groups each tractogram into clusters (using QuickBundles, DIPY) and registers the cluster **centroids**. Left: the atlas centroids (grey, MNI space) and ours (orange, this subject's native space) before registration. Right: ours after applying the affine found by SLR.

In [ ]:
plot_slr(atlas_centroids, subject_centroids, xfm)

**Steps 2–3: the thresholds.** For each model bundle, `reduction_thr` (mm) decides which clusters of our streamlines are close enough to the model to be considered at all, and `pruning_thr` (mm) how close each candidate streamline must be to the model, after the local registration, to be kept. Move the slider to see the left UF recognized with stricter (left) and looser (right) settings; the default is 15 / 7, and 20 / 10 and 25 / 12 are the looser examples in the DIPY tutorial. Larger thresholds give bigger, looser bundles.
(The first run takes a few minutes; results are cached.)

In [ ]:
THRESHOLDS = [(10, 5), (15, 7), (20, 10), (25, 12)]               # (reduction_thr, pruning_thr), mm
thr_dir = RECO / "thresholds"; thr_dir.mkdir(exist_ok=True)
rb_thr, uf_mni = None, {}
for red, prune in THRESHOLDS:
    f = thr_dir / f"lh.uf_{red}_{prune}.tck"
    if not f.exists():
        if rb_thr is None:                                            # our tractogram in atlas space, clustered once
            rb_thr = RecoBundles(transform_streamlines(wb, xfm), verbose=False, rng=np.random.RandomState(2001))
        t0 = time.time()
        recognized_mni, _ = rb_thr.recognize(model_bundle=models["lh.uf"], model_clust_thr=0.1, reduction_thr=red,
                                             pruning_thr=prune, reduction_distance="mdf", pruning_distance="mdf", slr=True)
        save_mm(recognized_mni, f)
        print(f"reduction {red} / pruning {prune}: {len(recognized_mni)} streamlines ({time.time() - t0:.0f} s)")
    uf_mni[(red, prune)] = load_mm(f)
plot_reco_thresholds(uf_mni, models["lh.uf"], "UF_L")

### From streamlines to density maps

To compare a set of streamlines with TRACULA's `path.pd.nii.gz`, to measure its volume, or to show it on a slice, we can turn a tract file into a visitation map: the **number of streamlines that pass through each voxel** (a track density image). In MRtrix3:

```bash
tckmap lh.af.tck lh.af_density.nii.gz -template dtifit_FA.nii.gz -upsample 2
```

* `-template`: make the map on the voxel grid of the FA image.
* `-upsample 2`: add a point halfway between consecutive streamline points before counting. Our streamlines have a 0.75 mm step (half the 1.5 mm voxel), but a streamline can still cut across the corner of a voxel between two points, and that voxel would not be counted. `tckmap` chooses this ratio by itself from the step size recorded in the file header (here 2); files written by DIPY, like our RecoBundles bundles, don't record it, so we set it explicitly to treat every bundle the same way.

DIPY's `density_map` counts only the voxels that *contain* a streamline point, i.e. it does not upsample. The cell below makes the maps with `tckmap` and compares the two for one bundle.

In [ ]:
#compute density maps
DENSITY = OUT / "density"; DENSITY.mkdir(exist_ok=True)
density = {m: {} for m in METHODS}
for b in BUNDLES:
    density["TRACULA"][b] = pd_vol[b]                                   # TRACULA's output already is a map
    for m in ("tckedit", "RecoBundles"):
        f = DENSITY / f"{m}_{b}.nii.gz"
        run(["tckmap", BUNDLE_DIR[m] / f"{b}.tck", f, "-template", FA_FILE, "-upsample", 2, "-force", "-quiet"],
            show=(m, b) == ("tckedit", "lh.af"))                        # echo the command once
        density[m][b] = nib.load(f).get_fdata()

# the same bundle with DIPY: only voxels that contain a streamline point are counted
d_mrtrix = density["tckedit"]["lh.af"]
d_dipy = density_map(bundles["tckedit"]["lh.af"], AFFINE, FA.shape)
#################################################################
print(f"tckedit lh.af  tckmap -upsample 2: {int((d_mrtrix > 0).sum()):,} voxels, max {d_mrtrix.max():.0f} streamlines per voxel")
print(f"               DIPY density_map:   {int((d_dipy > 0).sum()):,} voxels, max {d_dipy.max():.0f} streamlines per voxel")

### Summary: bundle volume and lengths

- The volume of a tract depends on where you threshold the map. Two options:
* **any voxel**: every voxel the map touches
* **voxels above some percentage**: For es TRACULA uses voxels above 20% of the map's *robust* maximum (99th percentile of its non-zero voxels). We apply the same rule to the outputs of RecoBundle and tckedit.

- Length: for tckedit and RecoBundles we give the median streamline length in mm. For TRACULA we use Len_Avg from pathstats.overall.txt, the average length of its path samples. TRACULA counts that length in voxels, because each path sample is stored as a chain of neighbouring voxels. So we multiply by the voxel size (1.5 mm) to get mm.

In [ ]:
vol = pd.DataFrame([dict(bundle=b, method=m,
                         **{"any voxel": (d > 0).sum() * VOXEL_MM3,
                            "> 20% of robust max": (read_pathstats(tracula_dir[b])["Volume"] if m == "TRACULA"   # TRACULA: pathstats
                                                    else above_robust_max(d).sum()) * VOXEL_MM3})
                    for b in BUNDLES for m in METHODS for d in [density[m][b]]])
vol = vol.pivot(index="bundle", columns="method").reindex(columns=METHODS, level=1).loc[BUNDLES]

vol.round(-1).astype(int)      # bundle volume in mm³

In [ ]:
# length: median streamline length in mm; TRACULA: Len_Avg from pathstats (in voxels) x voxel size
counts = pd.DataFrame([dict(bundle=b, method=m,
                            n_streamlines=len(bundles[m][b]) if m != "TRACULA" else np.nan,
                            length_mm=bundle_length(m, bundles[m][b], read_pathstats(tracula_dir[b]), VOXEL_MM))
                       for b in BUNDLES for m in METHODS])
counts.pivot(index="bundle", columns="method").reindex(columns=METHODS, level=1).loc[BUNDLES].round(0)

## 3. Look at the bundles

### 3.1 Interactive 3D viewer

* **Dropdown** (top left): pick a bundle.
* **Legend**: click a method to hide or show it; double-click to isolate it. For TRACULA, the default is `TRACULA path.pd`: a surface around its path probability distribution, which is what TRACULA computes. Click `TRACULA path.pd.trk` to show the path samples as streamlines instead.
* **Mouse**: drag to rotate, scroll to zoom, right-drag to pan.

At most 150 randomly chosen streamlines per method are drawn.

In [ ]:
plot_bundles_3d(bundles, pd_vol, brain, AFFINE, COLORS)

### 3.2 Side by side

The same bundle from each method in its own panel (each panel rotates independently). As above, TRACULA is shown as its path distribution; use the legend to switch to the path samples (`.trk`).

In [ ]:
plot_side_by_side(bundles, pd_vol, brain, AFFINE, COLORS)

### 3.3 Scroll through slices

The **density maps** from section 2 (streamlines per voxel; for TRACULA its own `path.pd.nii.gz`) on the FA map.
Every voxel a bundle touches is coloured; the colour is lighter where few streamlines (or path samples) pass and full where many do, relative to each map's robust maximum. The maps are on different scales, so compare *where* the bundles are, not their absolute values.

Move the slider to scroll through the slices. To look at another bundle or plane, change `bundle` and `plane` in the `slice_scroller` call and re-run the cell (bundles: `lh.af`, `rh.af`, `lh.cst`, `rh.cst`, `lh.ilf`, `rh.ilf`, `lh.uf`, `rh.uf`; planes: `axial`, `coronal`, `sagittal`).
The slider is plain Plotly, like the 3D views above: once the cell has run, it works without the kernel.

In [ ]:
slice_scroller(density, FA, COLORS, bundle="lh.cst", plane="coronal")    # try bundle="lh.cst", plane="coronal"

**What to look for**
* Compare tracts across methods:
    * Can you notice false positive or false negatives?
    * How do the definitions differ?

## 4. Mean FA per bundle

Common definitions:

| measure | what is averaged | MRtrix3 | DIPY | TRACULA (`pathstats`) |
|---|---|---|---|---|
| **voxel mean** | FA of every voxel the bundle touches, each voxel once | `tckmap` → `mrstats -mask` (voxels with density > 0) | `density = density_map(...)` → `FA[density > 0].mean()` | `FA_Avg`: `path.pd` voxels above 20% of its robust maximum |
| **voxel weighted** | FA of the same voxels, each weighted by its density (TRACULA: by `path.pd`) | `mrcalc FA density -mult`, then mean of that ÷ mean density (both `mrstats` over the bundle mask) | `(FA * density).sum() / density.sum()` | `FA_Avg_Weight`: same voxels, weighted by `path.pd` |
| **streamline mean** | mean FA along each streamline, then the mean across streamlines | `tcksample -stat_tck mean` → mean | `values_from_volume` → mean per streamline → mean | n/a |
| **point mean** | FA at every point of every streamline, all pooled (long streamlines count more) | `tcksample` → mean of all values | `values_from_volume` → mean of all values | n/a |

For TRACULA we use only values from `pathstats` outputs: its path samples (`.trk`) are not actual streamlines, so streamline- and point-based averages don't apply.

Weighted averages emphasise the bundle **core**, where most streamlines (or path samples) run. Unweighted averages give equal weight to sparse voxels at the bundle **periphery**.

Interpolation: `tcksample` and `values_from_volume` interpolate FA trilinearly at each streamline point, while voxel-based averages use the voxel values.

In [ ]:
STATS = OUT / "stats"; STATS.mkdir(exist_ok=True)

def fa_mrtrix(tck, tag):
    d = STATS / tag; d.mkdir(parents=True, exist_ok=True)
    show = tag == "tckedit_lh.af"                                   # echo the commands once
    run(["tcksample", tck, FA_FILE, d / "fa_per_streamline.txt", "-stat_tck", "mean", "-force", "-quiet"], show)
    per_sl = np.loadtxt(d / "fa_per_streamline.txt", comments="#").ravel()
    run(["tcksample", tck, FA_FILE, d / "fa_per_point.txt", "-force", "-quiet"], show)     # one row per streamline
    per_pt = np.concatenate([np.array(l.split(), float) for l in open(d / "fa_per_point.txt")
                             if l.strip() and not l.startswith("#")])
    run(["tckmap", tck, d / "density.nii.gz", "-template", FA_FILE, "-upsample", 2, "-force", "-quiet"], show)
    run(["mrcalc", d / "density.nii.gz", 0, "-gt", d / "mask.nii.gz", "-datatype", "bit", "-force", "-quiet"], show)
    run(["mrcalc", FA_FILE, d / "density.nii.gz", "-mult", d / "fa_x_density.nii.gz", "-force", "-quiet"], show)
    stat = lambda img: float(run(["mrstats", img, "-mask", d / "mask.nii.gz", "-output", "mean"], show))
    # weighted mean = sum(FA x density) / sum(density). mrstats has no weights and no sum, but over the same
    # N voxels mean(FA x density) / mean(density) = (sum/N) / (sum/N) = the same ratio
    return {"point mean": np.nanmean(per_pt), "streamline mean": np.nanmean(per_sl), "voxel mean": stat(FA_FILE),
            "voxel weighted": stat(d / "fa_x_density.nii.gz") / stat(d / "density.nii.gz")}

def fa_dipy(sl):
    vals = values_from_volume(FA, sl, AFFINE)                       # trilinear FA at every point
    density = density_map(sl, AFFINE, FA.shape)                    # streamlines per voxel
    return {"point mean": np.concatenate(vals).mean(),
            "streamline mean": np.mean([np.mean(v) for v in vals]),
            "voxel mean": FA[density > 0].mean(),
            "voxel weighted": (FA * density).sum() / density.sum()}

rows = []
for b in BUNDLES:
    ps = read_pathstats(tracula_dir[b])
    rows += [dict(bundle=b, source="TRACULA", tool="TRACULA pathstats", measure="voxel mean", FA=ps["FA_Avg"]),
             dict(bundle=b, source="TRACULA", tool="TRACULA pathstats", measure="voxel weighted", FA=ps["FA_Avg_Weight"])]
    for m in ("tckedit", "RecoBundles"):                            # streamline methods
        if len(bundles[m][b]) < 5:
            continue
        for meas, v in fa_mrtrix(BUNDLE_DIR[m] / f"{b}.tck", f"{m}_{b}").items():
            rows.append(dict(bundle=b, source=m, tool="MRtrix3", measure=meas, FA=v))
        for meas, v in fa_dipy(bundles[m][b]).items():
            rows.append(dict(bundle=b, source=m, tool="DIPY", measure=meas, FA=v))
fa_df = pd.DataFrame(rows)
fa_df.to_csv(OUT / "mean_fa_all_methods.csv", index=False)
display(fa_table(fa_df, ["voxel mean", "voxel weighted"], METHODS, BUNDLES))          # every method
display(fa_table(fa_df, ["streamline mean", "point mean"], METHODS, BUNDLES))         # streamline methods only

### Same bundle definition, different method: unweighted *vs* weighted mean FA

For the TRACULA bars we use TRACULA's own `pathstats` (`FA_Avg`, `FA_Avg_Weight`).
For RecoBundles and `tckedit` we use the MRtrix3 voxel-based values. Hover over a bar for the exact value.

In [ ]:
plot_mean_fa(fa_df, COLORS, BUNDLES)

**Take-home**
* When the definitions match, MRtrix3 and DIPY agree closely. The voxel-based averages differ a little because the two programs decide differently which voxels a streamline touches: `tckmap -upsample` also counts voxels a streamline only crosses, DIPY's `density_map` only voxels that contain one of its points.
* **The definition matters more than the software.** Weighted vs unweighted, and point- vs streamline- vs voxel-based averages, can shift mean FA by about as much as a typical group effect.
* **The bundle definition matters too.** For the same subject and the same FA map, the three segmentation methods give different values because they include different white matter, especially at the bundle ends.
* Whatever you choose, **report it** (e.g. "FA averaged over voxels intersected by the bundle, weighted by streamline density").

## 5. Along-tract profiles

A single mean hides *where* along the bundle FA differs. Tract profiling (e.g. AFQ, Yeatman et al. 2012; BUAN, Chandio et al. 2020) samples FA at a fixed number of positions along the bundle and averages across streamlines **node by node**.
Node-by-node averaging only works if:

1. All streamlines in the same direction - **Orientation** . Tractography does not have a "start": half the streamlines may run frontal→temporal, the other half temporal→frontal.
2. All streamline have the same number of points - **Resample**. Streamlines have different lengths and numbers of points. After resampling node *i* is the same for all streamlines.

After these steps we can:
1. **Sample** FA at every node.
2. **Weight** the streamlines at each node (optional). Streamlines far from the bundle core get less weight.
5. **Average** across streamlines at each node
6. **trim the ends**, where streamlines enter grey matter and partial-volume effects dominate (optional).

We first go through these steps for one bundle (left AF from `tckedit`), then apply them to every bundle and method.
DIPY has handy built-in functions to do this. Check the [tutorial](https://docs.dipy.org/dev/examples_built/streamline_analysis/afq_tract_profiles.html)

In [ ]:
START_ROI = {}                                               # ROI A voxels (mm): every profile starts at ROI A
for b in BUNDLES:
    roi = nib.load(roi_dir / f"{b}_A.nii.gz").get_fdata() > 0
    START_ROI[b] = nib.affines.apply_affine(AFFINE, np.argwhere(roi))   # voxel (i, j, k) -> mm
START_LABEL = {"af": "frontal", "cst": "cortex", "ilf": "occipital", "uf": "frontal"}
END_LABEL = {"af": "temporal", "cst": "brainstem", "ilf": "temporal", "uf": "temporal"}

# orient() (helpers.py) flips every streamline whose last point is closer to ROI A than its first point.
# It uses the distance to the nearest ROI A voxel, not to the ROI centre: ROI A is large, and an end can be
# inside it but far from its centre.
demo_b, demo_m = "lh.af", "tckedit"
raw = bundles[demo_m][demo_b]
oriented = orient(raw, START_ROI[demo_b])
n_flipped = sum(not np.allclose(a[0], b_[0]) for a, b_ in zip(raw, oriented))
print(f"{demo_m} {demo_b}: {len(raw)} streamlines, {n_flipped} had to be flipped")
print("points per streamline: min", min(len(s) for s in raw), "max", max(len(s) for s in raw))

**Step 1: orientation.** Below, the first point of every streamline is drawn as a dot, before and after orienting. Before, the "starts" are spread over both ends of the bundle. After, they are all at the frontal end.

In [ ]:
plot_orientation(raw, oriented, COLORS[demo_m])

**Steps 2–5.** Resample, sample, weight and average. The panels show what goes wrong if a step is skipped:
* **(a)** Without resampling, profiles of different streamlines cannot be averaged node by node because they have different lengths.
* **(b)** Without orientation, flipped streamlines get averaged against their mirror image, which blurs the profile and pushes it towards symmetry.
* **(c)** With Gaussian weighting, outlier streamlines count less, so the profile moves towards the bundle core.

In [ ]:
# step 2: resample to N_POINTS, step 3: sample FA
N_POINTS = 100
res_raw = set_number_of_points(raw, N_POINTS)
res_or = set_number_of_points(oriented, N_POINTS)
#Step 3 sample FA
fa_raw = np.array(values_from_volume(FA, res_raw, AFFINE))        
fa_or = np.array(values_from_volume(FA, res_or, AFFINE))
# step 4: Gaussian weights from the Mahalanobis distance to the bundle core, node by node
w = gaussian_weights(oriented, n_points=N_POINTS)                   
# step 5: average
prof_unoriented, prof_oriented, prof_weighted = fa_raw.mean(0), fa_or.mean(0), (w * fa_or).sum(0)

# the same thing in one call in DIPY
prof_afq = afq_profile(FA, oriented, AFFINE, n_points=N_POINTS, weights=w)
print("step by step == afq_profile:", np.allclose(prof_afq, prof_weighted, atol=1e-6))

x = np.linspace(0, 100, N_POINTS)                                   # node position, % along the tract
plot_profile_steps(values_from_volume(FA, raw[:40], AFFINE), fa_or, prof_unoriented, prof_oriented, prof_weighted,
                   COLORS[demo_m], along=f"{START_LABEL['af']} → {END_LABEL['af']}")

### 5.1 Profiles for every bundle and method

Resampling to a fixed number of points (above) puts node *i* at *i* % of **each streamline's own length**, so nodes only line up if all streamlines start and end at the same place, which they don't. Each tool has its own way to define node *i* by its **position** along a reference line instead. We use each method's own approach:

* **`tckedit` → MRtrix3 `tckresample -arc`.** The reference line is the bundle's mean line (oriented, resampled, averaged). Through three of its points (start, mid, end) `-arc 100` fits a circular arc, puts 100 planes perpendicular to it, and resamples every streamline where it crosses them: node *i* is the crossing with plane *i*, so the output is also oriented. `tcksample` then gives FA at every node and we average across streamlines.
  **Streamlines that do not cross every plane are dropped.** Many don't reach the planes at the bundle ends, so we start and end the arc 10 % inside the mean line (`TRIM`) and print how many streamlines are kept. (`-line` does the same with a straight line, for straight bundles.)
* **RecoBundles → DIPY BUAN** (Chandio et al. 2020). The reference line is the centroid of the atlas model bundle that RecoBundles used, moved into subject space with the inverse of the SLR transform from 2.3. `assignment_map` labels every point of every streamline with its nearest of the 100 centroid points, and FA is averaged over all points with the same label. No resampling, no streamline is dropped, no weighting.
* **TRACULA**: `pathstats.byvoxel.txt` already gives FA along the path. Each row is one voxel of the reference path (`path.ref.txt`, the mean path of the training subjects mapped to this subject). `FA` is the value in that voxel, and `FA_Avg` the mean, over the 1,500 path samples, of the FA at each sample's closest point. Because the reference path is the same for every subject, a bundle has the **same number of rows, in the same order, in every subject** (e.g. 95 for the left AF in both tutorial subjects), so row *i* is the same position along the tract and TRACULA's group tables need no resampling. The number of rows differs between bundles (72–96 here). To plot it on the same 0–100 % axis, `profile_tracula` (helpers.py) orients it (start at ROI A) and resamples it to 100 nodes.

In [ ]:
PROF = OUT / "profiles"; PROF.mkdir(exist_ok=True)
N_NODES = 100
TRIM = 10                                       # tckresample -arc: start and end the arc 10 % inside the mean line

def mean_line(sl, roi_xyz, n_points):
    # A bundle's mean line: orient, resample every streamline to n_points, average point by point.
    return np.mean(set_number_of_points(orient(sl, roi_xyz), n_points), axis=0)

def profile_arc(b, line):
    # MRtrix3: resample along an arc through 3 points of the mean line (101 points: point k is at k %), sample FA.
    start, mid, end = (",".join(f"{v:.2f}" for v in line[k]) for k in (TRIM, 50, 100 - TRIM))
    r, t = PROF / f"tckedit_{b}_arc.tck", PROF / f"tckedit_{b}_arc_fa.txt"
    show = b == "lh.af"
    run(["tckresample", BUNDLE_DIR["tckedit"] / f"{b}.tck", r, "-arc", N_NODES, start, mid, end, "-force", "-quiet"], show)
    run(["tcksample", r, FA_FILE, t, "-force", "-quiet"], show)
    rows = [list(map(float, l.split())) for l in open(t) if l.strip() and not l.startswith("#")]
    rows = np.array([v for v in rows if len(v) == N_NODES])          # streamlines that crossed every plane
    return rows.mean(0), len(rows)

def profile_buan(sl, centroid):
    # DIPY BUAN: label every point with its nearest centroid point (node), then average FA per node.
    nodes = assignment_map(sl, Streamlines([centroid]), N_NODES)
    fa = np.concatenate(values_from_volume(FA, sl, AFFINE))         # FA at every point, same order as nodes
    return np.bincount(nodes, fa, N_NODES) / np.bincount(nodes, minlength=N_NODES)

profiles, kept = [], []                                             # one row per bundle, method and node
for b in BUNDLES:
    rows = [("TRACULA", "TRACULA pathstats (FA_Avg)", np.linspace(0, 100, N_NODES),
             profile_tracula(read_byvoxel(tracula_dir[b]), START_ROI[b], AFFINE, N_NODES))]
    if len(bundles["tckedit"][b]) >= 5:
        prof, n = profile_arc(b, mean_line(bundles["tckedit"][b], START_ROI[b], 101))
        kept.append(dict(bundle=b, streamlines=len(bundles["tckedit"][b]), kept_by_arc=n))
        rows.append(("tckedit", "MRtrix3 tckresample -arc", np.linspace(TRIM, 100 - TRIM, N_NODES), prof))
    if len(bundles["RecoBundles"][b]) >= 5:
        centroid = mean_line(transform_streamlines(models[b], np.linalg.inv(xfm)), START_ROI[b], N_NODES)
        rows.append(("RecoBundles", "DIPY BUAN", np.linspace(0, 100, N_NODES), profile_buan(bundles["RecoBundles"][b], centroid)))
    for source, tool, pct, prof in rows:
        profiles += [dict(bundle=b, source=source, tool=tool, node=n, pct=p, FA=v) for n, (p, v) in enumerate(zip(pct, prof))]
prof_df = pd.DataFrame(profiles)
prof_df.to_csv(OUT / "along_tract_profiles.csv", index=False)
pd.DataFrame(kept).set_index("bundle")                              # tckedit streamlines kept by tckresample -arc

In [ ]:
plot_profiles(prof_df, COLORS, START_LABEL, END_LABEL)

**What to look for**
* **The overall shape agrees** across methods
* **"% along" is still not exactly the same anatomical point for every method.** Each method uses its own reference line (the `tckedit` mean line, the atlas centroid, TRACULA's reference path), and the `tckedit` profile only covers 10–90 % of its mean line. To compare methods (or subjects) node by node, use **one** reference for all of them, e.g. assign every bundle's points to the same centroid (BUAN), or cut all bundles with the same planes (`tckresample -line/-arc` with the same three points).

## 7. Bonus: from the whole-brain tractogram to a connectome

The same whole-brain tractogram can also be summarised as a **structural connectome**: divide the brain into regions (nodes) and look at the streamlines that connect each pair of regions (edges).

1. **Parcellation**: `labelconvert` turns the FreeSurfer `aparc+aseg` labels into **84 consecutive nodes** (68 Desikan-Killiany cortical regions + 16 subcortical structures). We use `aparc+aseg.bbr.nii.gz`, which is already in diffusion space.
   It needs two label tables that come with the software: FreeSurfer's `FreeSurferColorLUT.txt` (label number → name, in `$FREESURFER_HOME`) and MRtrix3's `fs_default.txt` (name → node 1–84, in `<MRtrix3>/share/mrtrix3/labelconvert/`). Without FreeSurfer (e.g. on Colab) we use the copy of `FreeSurferColorLUT.txt` included in the tutorial data.
   `labelconvert` matches regions by **name**. Old MRtrix3 versions (e.g. 3.0 from 2017) ship an `fs_default.txt` that only knows `Left-Thalamus-Proper`, the name used before FreeSurfer 7; with a FreeSurfer 7 segmentation both thalami then come out empty, without any error.
2. **`tck2connectome`**: each streamline is assigned to the node at each of its two ends (searching up to 4 mm from the endpoint, `-assignment_radial_search`). `-symmetric` makes the matrix symmetric, `-zero_diagonal` drops streamlines that start and end in the same region. `-out_assignments` saves the two nodes of every streamline.

** SIFT (or SIFT2)** Especially when building connectivity matrices that measure streamline count, it is improtant to try improve the biological interpretbilities of these measures. Both methods try to match streamline density to the fiber density of that voxel as astimated by CSD FODs. 
- Smith et al. (2013). SIFT: Spherical-deconvolution informed filtering of tractograms. NeuroImage, 67, 298–312.
- Smith et al. (2015). SIFT2: Enabling dense quantitative assessment of brain white matter connectivity using streamlines tractography. NeuroImage, 119, 338–351.

Our `wholebrain.tck` already went through **SIFT**, but here is an example call for SIFT2.
```bash
tcksift2 -act 5tt.mif tracks_10M.tck wmfod_norm.mif sift2_weights.txt -out_mu sift2_mu.txt
tck2connectome tracks_10M.tck DK_parcels.mif connectome.csv -tck_weights_in sift2_weights.txt \
               -symmetric -zero_diagonal -out_assignments dk_assignments.txt
```
(`-out_mu` saves the proportionality coefficient that converts the weighted counts to fibre density units.)

In [ ]:
CONN = OUT / "connectome"; CONN.mkdir(exist_ok=True)

# label tables from the installed software (find_lut in helpers.py prints where it found them)
fs_lut = find_lut("FreeSurferColorLUT.txt", TUTORIAL, CONN)    # FreeSurfer: label number -> name
node_lut = find_lut("fs_default.txt", TUTORIAL, CONN)          # MRtrix3: name -> node 1-84
#Generate connectome
parcels = CONN / "nodes.nii.gz"
run(["labelconvert", APARC_FILE, fs_lut, node_lut, parcels, "-force", "-quiet"])
run(["tck2connectome", WHOLEBRAIN_FILE, parcels, CONN / "connectome.csv", "-symmetric", "-zero_diagonal",
     "-out_assignments", CONN / "assignments.txt", "-force", "-quiet"])
        
C = np.loadtxt(CONN / "connectome.csv", delimiter=",")
n_pairs = C.shape[0] * (C.shape[0] - 1) // 2
print(f"{C.shape[0]} regions; {int((np.triu(C, 1) > 0).sum()):,} of {n_pairs:,} region pairs are connected")

In [ ]:
names = read_node_names(node_lut)
plot_connectome(C, names)      # node order: left cortex (1-34), left subcortical (35-42), right subcortical (43-49), right cortex (50-83), right cerebellum (84)

In [ ]:
# take the 10 strongest connections
# One row per pair of regions. The matrix is symmetric (C[a, b] == C[b, a]), so each pair is taken once (b > a).
# Matrix rows/columns start at 0, node numbers (in names) start at 1.
rows = []
for a in range(len(C)):
    for b in range(a + 1, len(C)):
        rows.append((names[a + 1], names[b + 1], int(C[a, b])))
edges = pd.DataFrame(rows, columns=["region_1", "region_2", "streamlines"])

edges.nlargest(10, "streamlines")

**Take a look at what those streamlines look like!** `tck2connectome -out_assignments` saved the two regions of every streamline (`assignments.txt`), so we can pull out the streamlines behind any connection with `connectome2tck`:

```bash
connectome2tck wholebrain.tck assignments.txt edge_ -nodes 21,23 -exclusive
```

* `-nodes 21,23`: only streamlines that involve these regions 
* `-exclusive`: only streamlines that connect *exactly* these two regions, here saved as `edge_21-23.tck` (one file per connection is the default)

We look at two connections: the **strongest** one (left precentral – postcentral, a short U-fibre between neighbouring gyri) and a **long-range** one (left caudal middle frontal – middle temporal, part of the arcuate fasciculus we dissected in section 2). To look at another pair, change the region names in `EDGES`.

In [ ]:
# streamlines of two connections, from the assignments saved by tck2connectome:
# the strongest one (a short U-fibre) and a long-range one that is part of the arcuate fasciculus
node = {name: n for n, name in names.items()}                   # region name -> node number (1-84)
EDGES = [("ctx-lh-precentral", "ctx-lh-postcentral"), ("ctx-lh-caudalmiddlefrontal", "ctx-lh-middletemporal")]
edge_tracts = {}
for a, b in EDGES:
    n1, n2 = sorted((node[a], node[b]))          # connectome2tck names the file edge_<smaller>-<larger>.tck
    run(["connectome2tck", WHOLEBRAIN_FILE, CONN / "assignments.txt", CONN / "edge_",
         "-nodes", f"{n1},{n2}", "-exclusive", "-force", "-quiet"])
    sl = load_tractogram(str(CONN / f"edge_{n1}-{n2}.tck"), str(FA_FILE), bbox_valid_check=False).streamlines
    edge_tracts[(n1, n2)] = sl
plot_edges(edge_tracts, parcels, names, brain, AFFINE)


## 6. Summary

| step | decision you have to make (and report) |
|---|---|
| bundle extraction | ROI-based (which ROIs? dilated? endpoints only?) / TRACULA / atlas-based (RecoBundles) |
| whole-tract FA | voxel vs streamline vs point averages; weighted or not; interpolation |
| along-tract FA | orientation rule, number of nodes, weighting, end trimming, clipping to common landmarks |

All outputs are in `outputs/`: bundles as `.tck` per method, the ROIs, `mean_fa_all_methods.csv` and `along_tract_profiles.csv`.

### References
* Yendiki et al. (2011) *Automated probabilistic reconstruction of white-matter pathways in health and disease using an atlas of the underlying anatomy.* Front Neuroinform.
* Maffei et al. (2021) *Using diffusion MRI data acquired with ultra-high gradient strength to improve tractography in routine-quality data.* NeuroImage.
* Garyfallidis et al. (2018) *Recognition of white matter bundles using local and global streamline-based registration and clustering.* NeuroImage.
* Yeh et al. (2018) *Population-averaged atlas of the macroscale human structural connectome and its network topology.* NeuroImage.
* Tournier et al. (2019) *MRtrix3: A fast, flexible and open software framework for medical image processing and visualisation.* NeuroImage.
* Smith et al. (2012) *Anatomically-constrained tractography*; Smith et al. (2013) *SIFT.* NeuroImage.
* Yeatman et al. (2012) *Tract profiles of white matter properties: automating fiber-tract quantification.* PLoS ONE.
* Chandio et al. (2020) *Bundle analytics, a computational framework for investigating the shapes and profiles of brain pathways across populations.* Sci Rep.